In [ ]:
import os
os.makedirs('/kaggle/working/proj', exist_ok=True)
open('/kaggle/working/proj/__init__.py', 'w').close()
open('/kaggle/working/proj/ckpt.py', 'w').write('"""Weights, resume state and the completion marker — three files, one atomic round.\n\nThe per-round weights file holds WEIGHTS ONLY and loads with weights_only=True: the\naggregated proxy w_bar_r and every client\'s personalized model w_s^k as plain state_dict\ntensors. RNG and the round counter live in a separate resume bundle keyed by the same\nround, so a reader never has to unpickle arbitrary objects to look at a checkpoint.\n\nPersistent state of the method is the round, w_bar_r and the N personalized models. The\nper-client proxies w_r^k are not kept: each is replaced by w_bar_r at the start of the\nnext round. AdamW is re-created per client per round (owner\'s decision, see nilm.py), so\nno optimizer state exists at a round boundary.\n"""\nimport csv, hashlib, json, os, random, shutil\nfrom pathlib import Path\nimport numpy as np, torch\n\nSUBDIRS = ("weights", "resume", "complete", "metrics", "preds", "confusion", "reports", "logs")\n\n# What a later session imports. `preds` and `logs` are not needed to CONTINUE, but a run\n# split across sessions must still be verifiable from its final output alone.\nRESUME_SUBDIRS = ("weights", "resume", "metrics", "confusion", "preds", "logs")\n\n# A run tree may start at a round r0 > 1 when the previous sessions were handed over as a\n# HANDOFF BUNDLE: only round r0\'s artifacts, plus this file, which carries the sha256 of\n# every weights file 1..r0 taken from the full tree the owner holds locally. Round r0 is\n# then anchored two ways — its own bytes must hash to chain[r0], and its prev_sha must be\n# chain[r0-1] — so a bundle cannot be assembled from two runs of the same config, and the\n# full chain is re-verified locally after the sessions are merged (merge_sessions.py).\nHANDOFF = "reports/handoff.json"\n\n# Every input that changes what the numbers mean. `rounds` IS in the list since the\n# per-round LR schedule (proj.nilm.lr_at) spans the whole run: the weights at round r\n# depend on how many rounds were planned, so a continuation push must plan the same\n# total. Paths, world_size, compile, eval_batch, max_seconds and require_resume are\n# operational and absent.\nFINGERPRINT_KEYS = (\n    "patch_len", "stem_ch", "dense_growth", "dense_layers", "incep_modules",\n    "fire_modules", "dropout", "num_classes", "n_features",\n    "lr", "lr_schedule", "lr_min", "rounds", "weight_decay", "clip",\n    "n_clients", "batch", "local_epochs", "seed",\n    "data_id", "run_name",\n)\n\n\ndef run_dir(run_name, base="/kaggle/working/runs"):\n    d = Path(base) / run_name\n    for s in SUBDIRS: (d / s).mkdir(parents=True, exist_ok=True)\n    return d\n\n\ndef fingerprint(cfg):\n    """A missing key is a KeyError, never a default. Silently hashing `None` for a key that\n    was renamed is exactly how a fingerprint stops protecting anything."""\n    missing = [k for k in FINGERPRINT_KEYS if k not in cfg]\n    if missing:\n        raise KeyError(f"fingerprint needs {missing} in CFG; add them, do not default them")\n    payload = json.dumps({k: cfg[k] for k in FINGERPRINT_KEYS}, sort_keys=True, default=str)\n    return hashlib.sha256(payload.encode()).hexdigest()[:16]\n\n\ndef file_sha(path):\n    """Hash of the file as written. Not a re-serialisation: torch.save embeds a zip whose\n    bytes are not reproducible, so only the bytes on disk are a stable identity."""\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for chunk in iter(lambda: f.read(1 << 20), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef atomic_save(obj, path):\n    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")\n    torch.save(obj, tmp); os.replace(tmp, path)     # replace is atomic on POSIX\n\n\ndef atomic_np_save(path, arr):\n    """np.save appends .npy to a name that lacks it, so write through a handle."""\n    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")\n    with open(tmp, "wb") as f:\n        np.save(f, arr); f.flush(); os.fsync(f.fileno())\n    os.replace(tmp, path)\n\n\n# --- RNG kept as tensors and primitives so the resume bundle also loads weights_only=True.\ndef rng_state():\n    npy = np.random.get_state()\n    return {"python": random.getstate(),\n            "numpy": (npy[0], torch.from_numpy(npy[1].copy()), int(npy[2]), int(npy[3]),\n                      float(npy[4])),\n            "torch": torch.get_rng_state(),\n            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}\n\n\ndef set_rng_state(s):\n    random.setstate(tuple(s["python"]))\n    n = s["numpy"]\n    np.random.set_state((n[0], n[1].numpy().astype(np.uint32), n[2], n[3], n[4]))\n    torch.set_rng_state(s["torch"].cpu())\n    if s.get("cuda") is not None: torch.cuda.set_rng_state_all(s["cuda"])\n\n\ndef cpu_sd(model):\n    """A detached CPU copy of a module\'s state_dict: tensors only, so the file it goes\n    into loads with weights_only=True."""\n    core = getattr(model, "_orig_mod", model)                 # unwrap torch.compile\n    return {k: v.detach().cpu().clone() for k, v in core.state_dict().items()}\n\n\n# --------------------------------------------------------------------------- write\ndef save_round_weights(global_sd, clients_sd, rnd, cfg, metrics, global_metrics, d):\n    """global_sd: state_dict of w_bar_r^t; clients_sd: {cid: state_dict of w_s^k at\n    round t} for ALL N clients. `metrics` is the mean over clients of the 10 metrics,\n    `global_metrics` the 10 metrics of w_bar_r. The caller writes the marker, and only\n    after every other artifact is on disk."""\n    prev = d / "weights" / f"round_{rnd - 1:03d}.pt"\n    # The hash of the weights this round was trained FROM. Two runs of the same config have\n    # the same fingerprint, so without this an import can keep round 1 of run A and take\n    # round 2 of run B and call the result one training history.\n    prev_sha = file_sha(prev) if rnd > 1 and prev.is_file() else None\n    atomic_save({"round": int(rnd),\n                 "global": global_sd,\n                 "clients": {int(c): sd for c, sd in sorted(clients_sd.items())},\n                 "cfg": {k: v for k, v in cfg.items()},       # rebuild recipe\n                 "fingerprint": fingerprint(cfg),\n                 "prev_sha": prev_sha,\n                 "metrics": metrics,\n                 "global_metrics": global_metrics},\n                d / "weights" / f"round_{rnd:03d}.pt")\n    # The RNG here is the DRIVER\'s, and the driver does not train. Recorded for forensics:\n    # worker training RNG is re-derived from (seed, round, client), so continuation does\n    # not depend on restoring this.\n    atomic_save({"round": int(rnd), "rng": rng_state(),\n                 "note": "no optimizer state: AdamW is re-created per client per round; "\n                         "worker RNG derives from (seed, round, client)",\n                 "fingerprint": fingerprint(cfg)},\n                d / "resume" / f"round_{rnd:03d}.pt")\n    return d / "weights" / f"round_{rnd:03d}.pt"\n\n\ndef mark_complete(d, rnd):\n    (d / "complete" / f"round_{rnd:03d}.done").write_text("")\n\n\n# --------------------------------------------------------------------------- rebuild\ndef _load_into(model, sd, expect_params):\n    sd = {k.removeprefix("module.").removeprefix("_orig_mod."): v for k, v in sd.items()}\n    bad = [k for k, v in sd.items() if v.is_floating_point() and not torch.isfinite(v).all()]\n    if bad:\n        raise RuntimeError(f"non-finite values in checkpoint tensors {bad[:3]}")\n    model.load_state_dict(sd, strict=True)\n    n = sum(p.numel() for p in model.parameters())\n    if expect_params is not None and n != expect_params:\n        raise RuntimeError(f"rebuilt {n:,} parameters, expected {expect_params:,}")\n    return model.eval()\n\n\ndef load_weights(path, build_model, expect_params=None, clients=None, device="cpu"):\n    """Rebuild w_bar_r and the personalized models at exactly this checkpoint, from\n    weights alone. Returns (R, {cid: S_k}, raw dict).\n\n    Every assert here exists because its failure is otherwise silent: strict=True catches a\n    filtered running_mean/var (eval() would then normalize by 0/1 and report nothing), the\n    parameter count catches a cfg that drifted from the one that trained these weights, and\n    the finite check catches a diverged tensor that argmax would turn into a plausible label.\n    `clients=None` rebuilds every client; pass a list to rebuild a subset."""\n    ck = torch.load(path, map_location="cpu", weights_only=True)\n    if ck.get("fingerprint") != fingerprint(ck["cfg"]):\n        raise RuntimeError("weights file fingerprint disagrees with its own cfg")\n    R = _load_into(build_model(ck["cfg"]), ck["global"], expect_params).to(device)\n    want = sorted(ck["clients"]) if clients is None else list(clients)\n    if sorted(ck["clients"]) != list(range(ck["cfg"]["n_clients"])):\n        raise RuntimeError(f"checkpoint holds clients {sorted(ck[\'clients\'])[:5]}..., "\n                           f"expected 0..{ck[\'cfg\'][\'n_clients\'] - 1}")\n    Ss = {int(c): _load_into(build_model(ck["cfg"]), ck["clients"][c], expect_params).to(device)\n          for c in want}\n    return R, Ss, ck\n\n\n# --------------------------------------------------------------------------- verify\ndef read_handoff(d):\n    """The handoff record of a tree that starts past round 1, or None. A malformed file\n    reads as absent, and an absent record means rounds before the first marker are simply\n    missing — which last_complete_round then treats as a gap, never as a start."""\n    p = Path(d) / HANDOFF\n    if not p.is_file():\n        return None\n    try:\n        h = json.loads(p.read_text())\n        h["round"] = int(h["round"])\n        h["chain"] = {int(k): str(v) for k, v in h["chain"].items()}\n        return h\n    except Exception:\n        return None\n\n\ndef handoff_record(d, rnd, fp, extra=None):\n    """The sha chain 1..rnd of a FULL tree, for a bundle that will carry round rnd alone.\n    Refuses a tree whose chain does not verify all the way from round 1. The caller writes\n    the dict to HANDOFF inside the bundle, never inside the full tree."""\n    d = Path(d)\n    if _first_round(d) != 1:\n        raise RuntimeError(f"{d} is itself a partial tree; export a handoff from the merged run")\n    last = last_complete_round(d, fp)\n    if last is None or last < rnd:\n        raise RuntimeError(f"{d}: rounds 1..{rnd} do not all verify (last={last}); "\n                           "nothing to hand off")\n    chain = {str(r): file_sha(d / "weights" / f"round_{r:03d}.pt") for r in range(1, rnd + 1)}\n    return {"round": int(rnd), "fingerprint": fp, "chain": chain, **(extra or {})}\n\n\ndef round_ok(d, rnd, fp=None, chain=True):\n    """A round counts only if every artifact of that round is present, READABLE, and links\n    to the round before it. The marker alone proves nothing. When the round before it is\n    not on disk, the link is checked against the handoff chain instead (see HANDOFF), and\n    the round\'s own bytes must match the chain too."""\n    w = d / "weights" / f"round_{rnd:03d}.pt"\n    r = d / "resume" / f"round_{rnd:03d}.pt"\n    m = d / "metrics" / f"round_{rnd:03d}.json"\n    c = d / "confusion" / f"round_{rnd:03d}.npy"\n    g = d / "confusion" / f"global_{rnd:03d}.npy"\n    for path in (w, r, m, c, g):\n        if not path.is_file() or path.stat().st_size == 0:\n            return False\n    try:\n        ck = torch.load(w, map_location="cpu", weights_only=True, mmap=True)\n        rs = torch.load(r, map_location="cpu", weights_only=True)   # not just "it exists"\n        row = json.loads(m.read_text())\n        np.load(c); np.load(g)\n    except Exception:\n        return False                      # truncated or corrupt reads as a failed round\n    if int(ck.get("round", -1)) != rnd or int(row.get("round", -1)) != rnd:\n        return False\n    if int(rs.get("round", -1)) != rnd:\n        return False\n    if fp is not None and (ck.get("fingerprint") != fp or rs.get("fingerprint") != fp):\n        return False\n    if chain:\n        # Round r is only meaningful as the product of round r-1. Same config, same\n        # fingerprint, different training history -> different bytes -> chain breaks here.\n        prev = d / "weights" / f"round_{rnd - 1:03d}.pt"\n        if rnd > 1 and not prev.is_file():\n            h = read_handoff(d)\n            if h is None or h["round"] != rnd or (fp is not None and h["fingerprint"] != fp):\n                return False                  # a bare round r > 1 is a gap, not a start\n            if h["chain"].get(rnd) != file_sha(w) or ck.get("prev_sha") != h["chain"].get(rnd - 1):\n                return False\n        else:\n            want = file_sha(prev) if rnd > 1 and prev.is_file() else None\n            if ck.get("prev_sha") != want:\n                return False\n    return True\n\n\ndef _markers(d):\n    return sorted(int(p.stem.split("_")[1]) for p in (d / "complete").glob("round_*.done"))\n\n\ndef _first_round(d):\n    """1 for a full tree; the handoff round for a bundle that starts later; the lowest\n    marker otherwise (round_ok then rejects it as a gap unless a handoff anchors it)."""\n    h = read_handoff(d)\n    m = _markers(d)\n    if h is not None and not (d / "weights" / f"round_{h[\'round\'] - 1:03d}.pt").is_file():\n        return h["round"]\n    return m[0] if m else 1\n\n\ndef last_complete_round(d, fp=None):\n    """Largest r such that rounds first..r are ALL complete, where first is 1 or the\n    handoff round of a bundle. A gap ends the run."""\n    last = 0\n    m = _markers(d)\n    top = m[-1] if m else 0\n    for r in range(_first_round(d), top + 1):\n        if not (d / "complete" / f"round_{r:03d}.done").is_file() or not round_ok(d, r, fp):\n            break\n        last = r\n    return last or None\n\n\n# --------------------------------------------------------------------------- resume\ndef _attached_source(run_name, attached=Path("/kaggle/input")):\n    if not attached.exists():\n        return None\n    roots = sorted({p.parent for p in attached.rglob("complete/round_*.done")\n                    if run_name in p.parts})\n    if len(roots) > 1:\n        raise RuntimeError(f"Multiple resume trees for {run_name}: {roots}")\n    return roots[0].parent if roots else None\n\n\ndef _import_from(src, d, fp):\n    """Copy through a staging tree, verify there, then publish one round at a time with its\n    marker last. Idempotent: a crash mid-publish leaves that round unmarked, and the next\n    attempt re-copies it from the still-mounted source. The source\'s own markers bound the\n    import, and the destination must not already hold a different history."""\n    stage = d.parent / f".{d.name}.import"\n    shutil.rmtree(stage, ignore_errors=True)\n    for sub in RESUME_SUBDIRS + ("complete", "reports"):\n        (stage / sub).mkdir(parents=True, exist_ok=True)\n    for sub in RESUME_SUBDIRS + ("complete",):\n        peer = src / sub\n        if not peer.is_dir(): continue\n        for f in peer.iterdir():\n            if f.is_file():\n                # copyfile, not copy2: a read-only mount\'s mode would carry across and the\n                # first rewrite would die with PermissionError.\n                shutil.copyfile(f, stage / sub / f.name)\n                os.chmod(stage / sub / f.name, 0o644)\n    if (src / HANDOFF).is_file():\n        shutil.copyfile(src / HANDOFF, stage / HANDOFF)\n        os.chmod(stage / HANDOFF, 0o644)\n\n    first = _first_round(stage)\n    src_last = first - 1\n    while ((stage / "complete" / f"round_{src_last + 1:03d}.done").is_file()\n           and round_ok(stage, src_last + 1, fp)):\n        src_last += 1\n    if src_last < first:\n        src_last = 0                                   # nothing verifiable, not even round first\n\n    if src_last and first > 1:\n        # The destination must not hold rounds the bundle cannot vouch for.\n        if any((d / "weights" / f"round_{r:03d}.pt").is_file() for r in range(1, first)):\n            shutil.rmtree(stage, ignore_errors=True)\n            raise RuntimeError(f"{d} already holds rounds before {first}; a handoff bundle "\n                               "cannot be spliced onto a tree that has its own history")\n        shutil.copyfile(stage / HANDOFF, d / HANDOFF)\n    for r in range(first, src_last + 1):\n        here = d / "weights" / f"round_{r:03d}.pt"\n        if here.is_file() and file_sha(here) != file_sha(stage / "weights" / f"round_{r:03d}.pt"):\n            shutil.rmtree(stage, ignore_errors=True)\n            raise RuntimeError(\n                f"round {r} in {d} and in {src} have the same config but different weights: "\n                "these are two different training runs, not one interrupted one. Refusing to "\n                "splice them. Detach one source, or start a new run_name.")\n\n    published = 0\n    for r in range(first, src_last + 1):\n        if not round_ok(d, r, fp):\n            for sub in RESUME_SUBDIRS:\n                for f in list((stage / sub).glob(f"round_{r:03d}.*")) + \\\n                         list((stage / sub).glob(f"global_{r:03d}.*")):\n                    shutil.copyfile(f, d / sub / f.name)\n                    os.chmod(d / sub / f.name, 0o644)\n        if not (d / "complete" / f"round_{r:03d}.done").is_file():\n            mark_complete(d, r)                     # marker last, per round\n        published = r\n    shutil.rmtree(stage, ignore_errors=True)\n    n_mark = len(list((src / "complete").glob("round_*.done"))) if (src / "complete").is_dir() else 0\n    print(f"[resume] {src}: {n_mark} marker(s), verified to round {src_last}, imported {first}..{published}"\n          + (f" (handoff bundle: chain 1..{first} attested, verified locally before export)"\n             if first > 1 else "")\n          if published else\n          f"[resume] {src} held no verifiable committed round; starting from 0")\n    return published\n\n\ndef resolve_resume(run_name, cfg=None, attached=Path("/kaggle/input"), d=None):\n    """working/ first, then any attached input (previous kernel output or a checkpoint\n    dataset). Returns the last round that is complete AND verified, or None."""\n    d = run_dir(run_name) if d is None else d\n    fp = fingerprint(cfg) if cfg is not None else None\n    src = _attached_source(run_name, attached)\n    if src is not None:\n        _import_from(src, d, fp)\n    rebuild_history(d, fp)\n    return last_complete_round(d, fp)\n\n\n# --------------------------------------------------------------------------- history\ndef _write_csv(p, rows):\n    tmp = Path(p).with_suffix(".csv.tmp")\n    with open(tmp, "w", newline="") as f:\n        w = csv.DictWriter(f, fieldnames=list(rows[0]))\n        w.writeheader(); w.writerows(rows)\n        f.flush(); os.fsync(f.fileno())\n    os.replace(tmp, p)                      # a crash mid-write cannot truncate the live file\n\n\ndef rebuild_history(d, fp=None):\n    """history.csv (mean over clients + global proxy per round) and clients.csv (one row\n    per client per round) are DERIVED from metrics/round_NNN.json, never authoritative.\n    Only the verified contiguous range 1..last goes in."""\n    last = last_complete_round(d, fp) or 0\n    rows, crow = [], []\n    for r in range(_first_round(d), last + 1):\n        p = d / "metrics" / f"round_{r:03d}.json"\n        if not p.is_file(): break\n        try: j = json.loads(p.read_text())\n        except Exception: break\n        rows.append({k: v for k, v in j.items() if k not in ("clients", "global")})\n        crow += [{"round": r, **{k: v for k, v in c.items() if k != "per_class"}}\n                 for c in j["clients"]]\n    if rows:\n        _write_csv(d / "history.csv", rows)\n        _write_csv(d / "clients.csv", crow)\n    else:\n        for n in ("history.csv", "clients.csv"):\n            if (d / n).exists(): (d / n).unlink()  # a stale CSV outlives the rounds it described\n    return len(rows)\n\n\ndef append_history(d, row, client_rows):\n    """Keyed by round: a redone round replaces its lines instead of duplicating them."""\n    p = d / "history.csv"\n    rows = {}\n    if p.exists():\n        with open(p) as f:\n            rows = {int(r["round"]): r for r in csv.DictReader(f)}\n    rows[int(row["round"])] = {k: str(v) for k, v in row.items()}\n    _write_csv(p, [rows[k] for k in sorted(rows)])\n    q = d / "clients.csv"\n    old = []\n    if q.exists():\n        with open(q) as f:\n            old = [r for r in csv.DictReader(f) if int(r["round"]) != int(row["round"])]\n    _write_csv(q, old + [{k: str(v) for k, v in c.items()} for c in client_rows])\n')


In [ ]:
import json, sys, shutil
from pathlib import Path
sys.path.insert(0, "/kaggle/working")
import proj.ckpt as C
RUN, EXPECT = 'nilm_20c', 33
inp = Path("/kaggle/input")
marks = sorted(p for p in inp.rglob("complete/round_*.done") if RUN in p.parts)
print("markers:", len(marks), "under", sorted({str(p.parent.parent) for p in marks}))
mf = [p for p in inp.rglob("reports/manifest.json") if RUN in p.parts]
assert len(mf) == 1, mf
m = json.loads(mf[0].read_text())
fp = C.fingerprint(m["cfg"])
print("fingerprint mounted", m["fingerprint"], "recomputed", fp)
assert fp == m["fingerprint"], "fingerprint mismatch: manifest cfg does not hash to its own fingerprint"
last = C.resolve_resume(RUN, m["cfg"])      # the production gate, same code, CPU
print("last verified round:", last)
assert last == EXPECT, f"expected {EXPECT}, got {last}"
shutil.rmtree("/kaggle/working/runs", ignore_errors=True)   # do not commit GBs of copies
print("PROBE_OK", RUN, last)
